In [ ]:
import pickle
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
from collections import defaultdict
import json
from scipy.stats import spearmanr, percentileofscore
import itertools
import seaborn as sns

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']
plt.rcParams['font.size'] = 16

%load_ext autoreload
%autoreload 2

num_heads = {'pythia-70m': 6*8, 'pythia-160m': 12*12, 'pythia-410m': 24*16, 'pythia-1b': 16*8,
    'pythia-1.4b': 24*16, 'pythia-2.8b': 32*32, 'pythia-6.9b': 32*32,
    'gpt2': 12*12, 'gpt2-medium': 24*16, 'gpt2-large': 36*20, 'gpt2-xl': 48*25, 'Llama-2-7b-hf': 32*32}

head_dim = {'pythia-70m': 512/8, 'pythia-160m': 768/12, 'pythia-410m': 1024/16, 'pythia-1b': 2048/8,
    'pythia-1.4b': 2048/16, 'pythia-2.8b': 2560/32, 'pythia-6.9b': 4096/32,
    'gpt2': 768/12, 'gpt2-medium': 1024/16, 'gpt2-large': 1280/20, 'gpt2-xl': 1600/25, 'Llama-2-7b-hf': 4096/32}

num_layers = {'pythia-70m': 6, 'pythia-160m': 12, 'pythia-410m': 24, 'pythia-1b': 16,
    'pythia-1.4b': 24, 'pythia-2.8b': 32, 'pythia-6.9b': 32,
    'gpt2': 12, 'gpt2-medium': 24, 'gpt2-large': 36, 'gpt2-xl': 48, 'Llama-2-7b-hf': 32}

num_params = {'pythia-70m': 70e6, 'pythia-160m': 160e6, 'pythia-410m': 410e6, 'pythia-1b': 1e9,
    'pythia-1.4b': 1.4e9, 'pythia-2.8b': 2.8e9, 'pythia-6.9b': 6.9e9,
    'gpt2': 117e6, 'gpt2-medium': 345e6, 'gpt2-large': 774e6, 'gpt2-xl': 1558e6, 'Llama-2-7b-hf': 7e9, 
    'Meta-Llama-3-8B': 8e9, 'Qwen2.5-7B:': 7e9}
# num_params = {k: np.log10(v) for k, v in num_params.items()}

shortnames = {'pythia-70m': '70M', 'pythia-160m': '160M', 'pythia-410m': '410M', 'pythia-1b': '1B',
    'pythia-1.4b': '1.4B', 'pythia-2.8b': '2.8B', 'pythia-6.9b': '6.9B',
    'gpt2': '117M', 'gpt2-medium': '345M', 'gpt2-large': '774M', 'gpt2-xl': '1.6B', 'Llama-2-7b-hf': '7B',
    'Meta-Llama-3-8B': '8B', 'Qwen2.5-7B:': '7BQ'}

longnames = {'pythia-70m': 'Pythia 70M', 'pythia-160m': 'Pythia 160M', 'pythia-410m': 'Pythia 410M', 'pythia-1b': 'Pythia 1B',
    'pythia-1.4b': 'Pythia 1.4B', 'pythia-2.8b': 'Pythia 2.8B', 'pythia-6.9b': 'Pythia 6.9B',
    'gpt2': 'GPT-2', 'gpt2-medium': 'GPT-2 Medium', 'gpt2-large': 'GPT-2 Large', 'gpt2-xl': 'GPT-2 XL', 'Llama-2-7b-hf': 'Llama 2 7B', 
    'Meta-Llama-3-8B': 'Llama 3 8B', 'Qwen2.5-7B': 'Qwen 2.5 7B'}

models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
model_names = list(map(lambda x: shortnames[x], models))


fv_paper_test = ['ag_news_val', 'capitalize_first_letter_val', 'conll2003_organization_val', 'national_parks_val', 'park-country_val', 
                    'person-occupation_val', 'animal_v_object_3_val', 'choose_first_of_5_val', 
                    'english-german_val', 'person-instrument_val', 'commonsense_qa_val']

new_tasks = ['capital_index_val',
    'capitalize_second_letter_val', 
    'abstract_clf_val',
    'french-english_val',
    'bind_fruit_val',
    'bind_capital_par_val',
    'bind_capital_val',
    'bind_shape_val']

test_data = list(map(lambda x: x.replace("_val",""), fv_paper_test + new_tasks))

In [ ]:
pythia_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']

In [ ]:
    
def top_heads(scores):
    heads = sorted(scores.keys(), key=lambda k: scores[k], reverse=True)
    return heads

def heads_over_threshold(scores, threshold):
    return list(
        filter(lambda x: scores[x] > threshold, scores.keys())
    )

def get_scores(model, head, ckpt=None):
    if ckpt is None:
        path = f"./outputs/heads/{model}"
    else:
        path = f"./outputs/heads/{model}-{ckpt}"
    if head == 'copy-induction':
        induction_head_path = f"{path}/induction.pkl"
        copying_head_path = f"{path}/copying.pkl"
        with open(induction_head_path, 'rb') as f:
            induction_heads = pickle.load(f)
        with open(copying_head_path, 'rb') as f:
            copying_heads = pickle.load(f)
        scores = {k: 1/3 * induction_heads[k] + 2/3 * copying_heads[k] for k in induction_heads.keys()}
    else:
        head_path = f"{path}/{head}.pkl"
        with open(head_path, 'rb') as f:
            scores = pickle.load(f)
    return scores

induction_heads = dict()

for model in models:
    path = f"./outputs/heads/{model}/induction.pkl"
    try:
        with open(path, 'rb') as f:
            induction_heads[model] = pickle.load(f)
    except FileNotFoundError:
        print(f"Model {model} does not have induction heads")

fv_heads = dict()

for model in models:
    path = f"./outputs/heads/{model}/fv.pkl"
    try:
        with open(path, 'rb') as f:
            fv_heads[model] = pickle.load(f)
    except FileNotFoundError:
        print(f"Model {model} does not have fv heads")

cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

In [ ]:


xtick_values = [num_params[model] for model in models]
xtick_values[3] = 340e6
xtick_values[4] = 450e6
xtick_values[7] = 1.4e9
xtick_values[8] = 1.8e9
xtick_values[10] = 6.5e9
xtick_values[11] = 8.2e9

In [ ]:
ih_scores = pd.DataFrame([(num_params[model], v) for model in models for v in induction_heads[model].values()], columns=['params', 'scores'])
fv_scores = pd.DataFrame([(num_params[model], v) for model in models for v in fv_heads[model].values()], columns=['params', 'scores'])
palette = sns.color_palette("pastel")

box_palette = {str(num_params[model]): palette[0] if 'pythia' in model else palette[3] if 'gpt2' in model else palette[2] for model in models}

In [ ]:
sns.set_palette("pastel")

plt.figure(figsize=(10, 6))
ax = sns.boxplot(x='params', y='scores', data=ih_scores, palette=box_palette)
# ax.set_xscale('log')
plt.xlabel('Size')
plt.ylabel('Induction score')
# ax.set_xticks(ticks=xtick_values)
ax.set_xticklabels(model_names, rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/ih_boxplot.pdf')
plt.show()

In [ ]:
sns.set_palette("pastel")

plt.figure(figsize=(10, 6))
ax = sns.boxplot(x='params', y='scores', data=fv_scores, palette=box_palette)
ax.set_yscale('linear')
plt.xlabel('Size')
plt.ylabel('FV score')
# ax.set_xticks(ticks=xtick_values)
ax.set_xticklabels(model_names, rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/fv_boxplot.pdf')
plt.show()

In [ ]:
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = False
val_only = True
def plot_task_ablations(model, i, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), test_data))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks
    if i == 19:
        task = tasks[18]
    else:
        task = tasks[i]

    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    all_copy = defaultdict(list)
    all_copy_ih = defaultdict(list)
    if not excl and i != 19:
        ax.set_title(task)
    if excl and i != 19:
        ax.set_title(f'{task}')
    

    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    copy_outputs_dir = outputs_dir
    if excl:
        outputs_dir += "_excltop2"
        copy_outputs_dir += "_excltop2"
    for num_ablate in num_ablates:
        bases = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        copyings = []
        copy_inductions = []

        try:
            with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                all_bases[model].append(np.mean(scores))
        except Exception as e:
            print(e)
            pass
        try:
            with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                all_inductions[model].append(np.mean(scores))
        except:
            pass
        try:
            with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                all_fvs[model].append(np.mean(scores))
        except Exception as e:
            print(e)
            pass
        try:
            with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                all_randoms[model].append(np.mean(scores))
        except:
            pass
        try:
            with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                all_rand_inductions[model].append(np.mean(scores))
        except:
            pass
            # try:
            #     with open(f"{copy_outputs_dir}/{task}_copying_{num_ablate}.txt", "r") as f:
            #         scores = list(map(float, f.readlines()))
            #         copyings.append(np.mean(scores))
            # except:
            #     pass
            # try:
            #     with open(f"{copy_outputs_dir}/{task}_copy-induction_{num_ablate}.txt", "r") as f:
            #         scores = list(map(float, f.readlines()))
            #         copy_inductions.append(np.mean(scores))
            # except:
            #     pass
        # print(len(inductions))
        # bases = np.mean([float(x) for x in bases])
        # inductions = np.mean([float(x) for x in inductions])
        # fvs = np.mean([float(x) for x in fvs])
        # randoms = np.mean([float(x) for x in randoms])
        # rand_inductions = np.mean([float(x) for x in rand_inductions])
        # copyings = np.mean([float(x) for x in copyings])
        # copy_inductions = np.mean([float(x) for x in copy_inductions])
    
        # all_bases[model].append(bases)
        # all_inductions[model].append(inductions)
        # all_fvs[model].append(fvs)
        # all_randoms[model].append(randoms)
        # all_rand_inductions[model].append(rand_inductions)
        # all_copy[model].append(copyings)
        # all_copy_ih[model].append(copy_inductions)
    if i != 19:
        ax.set_xlabel('Ablation (%)')
        # if model == 'pythia-410m':
        ax.set_ylabel('ICL Accuracy')
        p_ablates = [n * 100 for n in num_ablates]
        ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
        ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
        # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
        if excl:
            ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', linestyle=':', color=colors[2])
            # ax.plot(p_ablates, all_inductions[model], label=f'Induction (data)', marker='o', linestyle=':', color=colors[2])
            ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', linestyle=':', color=colors[3])
        else:
            ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2])
            # ax.plot(p_ablates, all_inductions[model], label=f'Induction (data)', marker='o', color=colors[2])
            ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])
            # ax.plot(p_ablates, all_copy[model], label=f'Copy', marker='x', color=colors[4])
            # ax.plot(p_ablates, all_copy_ih[model], label=f'Copy Induction', marker='x', linestyle=':', color=colors[4])
    else:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.xaxis.set_ticks([])
        ax.yaxis.set_ticks([])
        ax.plot([],[], label=f'Clean', color=colors[5])
        ax.plot([], [], label=f'Random', marker='', color=colors[0])
        # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
        # if excl:
        #     ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', linestyle=':', color=colors[4])
        #     ax.plot(p_ablates, all_inductions[model], label=f'Induction (data)', marker='o', linestyle=':', color=colors[2])
        #     ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', linestyle=':', color=colors[3])
        # else:
        ax.plot([], [], label=f'Induction', marker='o', color=colors[2])
        # ax.plot(p_ablates, all_inductions[model], label=f'Induction (data)', marker='o', color=colors[2])
        ax.plot([], [], label=f'FV', marker='x', color=colors[3])
    
for model in ['pythia-6.9b']:
  
    plt.figure()
    fig, axs = plt.subplots(4, 5, figsize=(20, 10))

    for ax, i in zip(axs.flat, range(20)):
        try:
            plot_task_ablations(model, i, ax)
        except Exception as e:
            print(e)
            print(model)
            ax.set_title(f'{model} missing')
    plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
    # plt.title(f'{model}')
    # plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.legend(loc='center left')
    plt.savefig(f'plots/abl_{model}.pdf')
    plt.show()
    # break

In [ ]:
plt.figure()

percent = False
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))
threshold = False

x_label = 'num_params'
if x_label == 'num_params':
    # x_values = [num_params[model] for model in models]
    x_axis = num_params
else:
    # x_values = [num_heads[model] for model in models]
    x_axis = head_dim
models = sorted(x_axis.keys(), key=lambda k: x_axis[k], reverse=False)

def plot_num_heads(models, head, ax): 

    x_values = []

    scores = defaultdict(list)

    # ax.set_title(head)
    ax.set_xlabel('Model size')
    
    for model in models:


        if 'induction' in head:
            t1, t2, t3 = 0.15, 0.2, 0.3
            ax.set_ylabel('Induction score')
        else:
            t1, t2, t3 = 1e-3, 2e-3, 3e-3
            ax.set_ylabel('FV score')
        
        if not threshold:
            t1 = max(2, 0.01 * num_heads[model])
            t2 = max(2, 0.02 * num_heads[model])
            t3 = max(2, 0.02 * num_heads[model])
            # # t1 = 5
            # t2 = 10
            # # t3 = 15
            # t3 = 10
        
        if head == 'copy-induction':
            induction_head_path = f"./outputs/heads/{model}/induction.pkl"
            copying_head_path = f"./outputs/heads/{model}/copying.pkl"
            if not os.path.exists(induction_head_path) or not os.path.exists(copying_head_path):
                continue
            with open(induction_head_path, 'rb') as f:
                induction_heads = pickle.load(f)
            with open(copying_head_path, 'rb') as f:
                copying_heads = pickle.load(f)
            heads = {k: 1/3 * induction_heads[k] + 2/3 * copying_heads[k] for k in induction_heads.keys()}
        else:
            head_path = f"./outputs/heads/{model}/{head}.pkl"
            if not os.path.exists(head_path):
                continue
            with open(head_path, 'rb') as f:
                heads = pickle.load(f)
        values = list(heads.values())
        mean = np.mean(values)
        median = np.median(values)
        
        if threshold:
            gt_02 = sum(np.array(values) > t1)
            gt_05 = sum(np.array(values) > t2)
            gt_07 = sum(np.array(values) > t3)
        else:
            heads = sorted(values, reverse=True)
            
            gt_02 = np.mean(heads[:int(t1)])
            gt_05 = np.mean(heads[:int(t2)])
            gt_07 = np.mean(heads[:int(t3)])

        if percent:
            total = len(values) / 100
        else:
            total = 1

        scores['mean'].append(mean)
        scores['max'].append(max(values))
        scores['gt_02'].append(gt_02/total)
        scores['gt_05'].append(gt_05/total)
        scores['gt_07'].append(gt_07/total)
        scores['num_heads'].append(x_axis[model])
        if 'pythia' in model:
            marker = 'o'
        elif 'gpt2' in model:
            marker = 'x'
        else:
            marker = 's'
        scores['marker'].append(marker)

        x_values.append(x_axis[model])
    
    if x_label == 'num_params':
        ax.set_xscale('log')
    
    ax.plot(x_values, scores['max'], color=colors[2], label=f'max')
    ax.plot(x_values, scores['gt_05'], color=colors[4], label=f'mean top 2%')
    ax.plot(x_values, scores['mean'], color=colors[3], label=f'mean')

    for x, s1, s2, s3, model in zip(x_values, scores['max'], scores['gt_05'], scores['mean'], models):
        ax.plot(x, s1, marker=f'${model[0].upper()}$', color=colors[2], markersize=10)
        ax.plot(x, s2, marker=f'${model[0].upper()}$', color=colors[4], markersize=10)
        ax.plot(x, s3, marker=f'${model[0].upper()}$', color=colors[3], markersize=10)
        


    # Getting handles and labels
    ax2 = plt.gca()
    handles, labels = ax2.get_legend_handles_labels()

    by_label = dict(zip(labels, handles))
    # ax.legend(by_label.values(), by_label.keys())
    
    ax.set_xticks(ticks=xtick_values)
    ax.set_xticklabels(model_names, rotation=45, ha='right')
    if 'induction' in head:
        ax.legend(fontsize='small')


plt.figure()

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax, head in zip(axs.flat, ['induction', 'fv']):
    plot_num_heads(models, head, ax)
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/head_strength.pdf')
plt.show()


In [ ]:

outputs_dir = f"./outputs/fv_scores/pythia-70m"
tasks = [x.replace(f'_clean.json', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_clean.json')]
all_clean = defaultdict(list)
all_zs_clean = defaultdict(list)
all_zs_intervention = defaultdict(list)
all_zs_random = defaultdict(list)
all_shuffled_clean = defaultdict(list)
all_shuffled_intervention = defaultdict(list)
all_shuffled_random = defaultdict(list)

tmp_models = []

for model in models:
    try:
        for task in tasks:
            outputs_dir = f"./outputs/fv_scores/{model}"
        
            clean = json.load(open(f"{outputs_dir}/{task}_clean.json", "r"))
            clean = np.mean([x['clean_topk'][0][1] for x in clean.values()])

            zs = json.load(open(f"{outputs_dir}/{task}_zs.json", "r"))
            zs_clean = zs['clean_topk'][0][1]
            zs_intervention = zs['intervention_topk'][0][1]
            zs_random = json.load(open(f"{outputs_dir}/{task}_zs_random.json", "r"))
            zs_random_clean = zs_random['clean_topk'][0][1]
            zs_random_intervention = zs_random['intervention_topk'][0][1]

            shuffled = json.load(open(f"{outputs_dir}/{task}_shuffled.json", "r"))
            shuffled_clean = shuffled['clean_topk'][0][1]
            shuffled_intervention = shuffled['intervention_topk'][0][1]
            shuffled_random = json.load(open(f"{outputs_dir}/{task}_shuffled_random.json", "r"))
            shuffled_random_clean = shuffled_random['clean_topk'][0][1]
            shuffled_random_intervention = shuffled_random['intervention_topk'][0][1]

            all_clean[task].append(clean)
            all_zs_clean[task].append(zs_clean)
            all_zs_intervention[task].append(zs_intervention)
            all_zs_random[task].append(zs_random_intervention)
            all_shuffled_clean[task].append(shuffled_clean)
            all_shuffled_intervention[task].append(shuffled_intervention)
            all_shuffled_random[task].append(shuffled_random_intervention)

        tmp_models.append(model)
    except Exception as e:
        print(e)
        print(model)


x_label = 'num_params'
if x_label == 'num_params':
    x_values = [num_params[model] for model in tmp_models]
    x_axis = num_params
else:
    x_values = [num_heads[model] for model in tmp_models]
    x_axis = num_heads

cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

scores = defaultdict(list)

plt.figure(figsize=(10, 5))
# plt.title('FV intervention')
plt.xlabel('Size')
plt.ylabel('Accuracy')

mean_clean = []
mean_zs_clean = []
mean_zs_intervention = []
mean_zs_random = []
mean_shuffled_clean = []
mean_shuffled_intervention = []
mean_shuffled_random = []

for i,model in enumerate(tmp_models):

    clean = np.mean([all_clean[task][i] for task in tasks if not np.isnan(all_clean[task][i])])
    zs_clean = np.mean([all_zs_clean[task][i] for task in tasks if not np.isnan(all_zs_clean[task][i])])
    zs_intervention = np.mean([all_zs_intervention[task][i] for task in tasks if not np.isnan(all_zs_intervention[task][i])])
    zs_random = np.mean([all_zs_random[task][i] for task in tasks if not np.isnan(all_zs_random[task][i])])
    shuffled_clean = np.mean([all_shuffled_clean[task][i] for task in tasks if not np.isnan(all_shuffled_clean[task][i])])
    shuffled_intervention = np.mean([all_shuffled_intervention[task][i] for task in tasks if not np.isnan(all_shuffled_intervention[task][i])])
    shuffled_random = np.mean([all_shuffled_random[task][i] for task in tasks if not np.isnan(all_shuffled_random[task][i])])
    # check for NaN
    if not np.isnan(clean):
        mean_clean.append(clean)
    if not np.isnan(zs_clean):
        mean_zs_clean.append(zs_clean)
    if not np.isnan(zs_intervention):
        mean_zs_intervention.append(zs_intervention)
    if not np.isnan(zs_random):
        mean_zs_random.append(zs_random)
    if not np.isnan(shuffled_clean):
        mean_shuffled_clean.append(shuffled_clean)
    if not np.isnan(shuffled_intervention):
        mean_shuffled_intervention.append(shuffled_intervention)
    if not np.isnan(shuffled_random):
        mean_shuffled_random.append(shuffled_random)
   
for x, c, s, f, r, m in zip(x_values, mean_clean, mean_shuffled_clean, mean_shuffled_intervention, mean_shuffled_random, tmp_models):
    plt.plot(x, c, marker=f'${m[0].upper()}$', color=colors[3], markersize=10)
    plt.plot(x, s, marker=f'${m[0].upper()}$', color=colors[2], markersize=10)
    plt.plot(x, f, marker=f'${m[0].upper()}$', color=colors[4], markersize=10)
    plt.plot(x, r, marker=f'${m[0].upper()}$', color=colors[0], markersize=10)

plt.plot(x_values, mean_clean, label='clean', marker='', color=colors[3])
plt.plot(x_values, mean_shuffled_clean, label='shuffled', marker='', color=colors[2])
plt.plot(x_values, mean_shuffled_intervention, label='shuffled+FV', marker='', color=colors[4])
plt.plot(x_values, mean_shuffled_random, label='shuffled+random', marker='', color=colors[0])
plt.xscale('log')

# Getting handles and labels
ax = plt.gca()
handles, labels = ax.get_legend_handles_labels()
ax.set_xticks(ticks=xtick_values)
ax.set_xticklabels(model_names, rotation=45, ha='right')

by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize='medium')
plt.tight_layout()

plt.savefig('plots/fv_intervention.pdf')
plt.show()

In [ ]:
sub_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']

plt.figure()

def plot_head_layers(model, ax):
    num_scores = max(min(int(np.ceil(num_heads[model] * 0.02)), 10), 3)

    for i,head in enumerate(['induction', 'fv']):
        head_path = f"./outputs/heads/{model}/{head}.pkl"

        if head == 'copy-induction':
            induction_head_path = f"./outputs/heads/{model}/induction.pkl"
            copying_head_path = f"./outputs/heads/{model}/copying.pkl"
            with open(induction_head_path, 'rb') as f:
                induction_heads = pickle.load(f)
            with open(copying_head_path, 'rb') as f:
                copying_heads = pickle.load(f)
            task_heads = {k: 1/3 * induction_heads[k] + 2/3 * copying_heads[k] for k in induction_heads.keys()}
        else:
            with open(head_path, "rb") as f:
                task_heads = pickle.load(f)
            
        high_heads = sorted(task_heads, key=lambda k: task_heads[k], reverse=True)[:num_scores]
        # Extracting x and y values from the dictionary
        x_values = [float(key.split('.')[0]) for key in high_heads]
        y_values = [task_heads[h] for h in high_heads]
        
        # Extract the second numbers from the keys
        second_numbers = [float(key.split('.')[1]) for key in high_heads]

        # Normalize the second numbers to lie between 0 and 1
        min_val = min(second_numbers)
        max_val = max(second_numbers)
        normalized = [(val - min_val) / (max_val - min_val) for val in second_numbers]


        # Plotting the scatter plot
        if head == 'fv':
            ax2 = ax.twinx()
            head = 'FV head'
            ax2.scatter(x_values, y_values, color=colors[3], marker='x', label=head)
            # weighted average x_values with y_values as weights
            x_mean = np.average(x_values, weights=y_values)
            # plot vertical line
            ax2.axvline(x_mean, color=colors[3], linestyle='--')
            if model == 'pythia-6.9b':
                ax2.set_ylabel('FV score (×)', color=colors[3])
        else:
            head = 'Induction head'
            ax.scatter(x_values, y_values, color=colors[2], marker='o', label=head)
            x_mean = np.average(x_values, weights=y_values)
            # plot vertical line
            ax.axvline(x_mean, color=colors[2], linestyle='--')
            if model == sub_models[0]:
                ax.set_ylabel('Induction score (•)', color=colors[2])
    ax.set_xlabel('Layer')
    ax.set_xlim(0, num_layers[model])
    ax.set_xticks(np.arange(0, num_layers[model], 1))
    plt.gca().set_xticklabels([str(i) if i % 5 == 0 else '' for i in range(num_layers[model])])

    
    ax.set_title(f"{longnames[model]}")
    # ax.legend()
    ax.grid(True)

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
# fig, axs = plt.subplots(3, 3, figsize=(14, 14))

for ax, model in zip(axs.flat, sub_models):
    try:
        plot_head_layers(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')

plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/head_layers.svg', transparent=True)
plt.show()

In [ ]:
plt.figure()

def plot_head_layers(model, ax):
    num_scores = max(min(int(np.ceil(num_heads[model] * 0.02)), 10), 3)

    for i,head in enumerate(['induction', 'fv']):
        head_path = f"./outputs/heads/{model}/{head}.pkl"

        if head == 'copy-induction':
            induction_head_path = f"./outputs/heads/{model}/induction.pkl"
            copying_head_path = f"./outputs/heads/{model}/copying.pkl"
            with open(induction_head_path, 'rb') as f:
                induction_heads = pickle.load(f)
            with open(copying_head_path, 'rb') as f:
                copying_heads = pickle.load(f)
            task_heads = {k: 1/3 * induction_heads[k] + 2/3 * copying_heads[k] for k in induction_heads.keys()}
        else:
            with open(head_path, "rb") as f:
                task_heads = pickle.load(f)
            
        high_heads = sorted(task_heads, key=lambda k: task_heads[k], reverse=True)[:num_scores]
        # Extracting x and y values from the dictionary
        x_values = [float(key.split('.')[0]) for key in high_heads]
        y_values = [task_heads[h] for h in high_heads]
        
        # Extract the second numbers from the keys
        second_numbers = [float(key.split('.')[1]) for key in high_heads]

        # Normalize the second numbers to lie between 0 and 1
        min_val = min(second_numbers)
        max_val = max(second_numbers)
        normalized = [(val - min_val) / (max_val - min_val) for val in second_numbers]


        # Plotting the scatter plot
        if head == 'fv':
            ax2 = ax.twinx()
            head = 'FV head'
            ax2.scatter(x_values, y_values, color=colors[3], marker='x', label=head)
            # weighted average x_values with y_values as weights
            x_mean = np.average(x_values, weights=y_values)
            # plot vertical line
            ax2.axvline(x_mean, color=colors[3], linestyle='--')
            ax2.set_ylabel('FV score (×)', color=colors[3])
        else:
            head = 'Induction head'
            ax.scatter(x_values, y_values, color=colors[2], marker='o', label=head)
            x_mean = np.average(x_values, weights=y_values)
            # plot vertical line
            ax.axvline(x_mean, color=colors[2], linestyle='--')
            ax.set_ylabel('Induction score (•)', color=colors[2])
    ax.set_xlabel('Layer')
    ax.set_xlim(0, num_layers[model])
    
    ax.set_title(f"{longnames[model]}")
    # ax.legend()
    ax.grid(True)

fig, axs = plt.subplots(4, 3, figsize=(13, 12))
# fig, axs = plt.subplots(3, 3, figsize=(14, 14))

for ax, model in zip(axs.flat, models):
    try:
        plot_head_layers(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')

plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/head_layers_all.pdf')
plt.show()

In [ ]:
plt.figure()
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))
percent = True
x_label = 'num_params'
percentile = 95
if x_label == 'num_params':
    # x_values = [num_params[model] for model in models]
    x_axis = num_params
else:
    # x_values = [num_heads[model] for model in models]
    x_axis = num_heads
models = sorted(x_axis.keys(), key=lambda k: x_axis[k], reverse=False)

def plot_over_threshold(models, head, ax, percentile=95, p_heads=0.05): 

    x_values = []

    scores = defaultdict(list)

    # ax.set_title(f'heads in top 2% with {head} score over {percentile}th')
    ax.set_xlabel('Size')
    if percent:
        ax.set_ylabel('Overlap (%)')
    else:
        ax.set_ylabel('Number')
    for model in models:
        n_heads = p_heads * num_heads[model]
        n_heads = np.ceil(n_heads).astype(int)
        # n_heads = np.min([10, num_heads[model]])

        ih_score = get_scores(model, "induction")
        rih_score = get_scores(model, "induction")
        fv_score = get_scores(model, "fv")

        top_ih = top_heads(ih_score)[:n_heads]
        top_rih = top_heads(rih_score)[:n_heads]
        top_fv = top_heads(fv_score)[:n_heads]

        ih_scores = sorted(list(ih_score.values()))
        rih_scores = sorted(list(rih_score.values()))
        fv_scores = sorted(list(fv_score.values()))

        ih_threshold = np.percentile(ih_scores, percentile)
        rih_threshold = np.percentile(rih_scores, percentile)
        fv_threshold = np.percentile(fv_scores, percentile)

        ih_ih_num = len([h for h in top_ih if ih_score[h] > ih_threshold])
        ih_rih_num = len([h for h in top_ih if rih_score[h] > rih_threshold])
        ih_fv_num = len([h for h in top_ih if fv_score[h] > fv_threshold])
        rih_ih_num = len([h for h in top_rih if ih_score[h] > ih_threshold])
        rih_rih_num = len([h for h in top_rih if rih_score[h] > rih_threshold])
        rih_fv_num = len([h for h in top_rih if fv_score[h] > fv_threshold])
        fv_ih_num = len([h for h in top_fv if ih_score[h] > ih_threshold])
        fv_rih_num = len([h for h in top_fv if rih_score[h] > rih_threshold])
        fv_fv_num = len([h for h in top_fv if fv_score[h] > fv_threshold])

        if percent:
            ih_ih_num = ih_ih_num / n_heads
            ih_rih_num = ih_rih_num / n_heads
            ih_fv_num = ih_fv_num / n_heads
            rih_ih_num = rih_ih_num / n_heads
            rih_rih_num = rih_rih_num / n_heads
            rih_fv_num = rih_fv_num / n_heads
            fv_ih_num = fv_ih_num / n_heads
            fv_rih_num = fv_rih_num / n_heads
            fv_fv_num = fv_fv_num / n_heads

        scores['rih_rih_score'].append(rih_rih_num*100)
        scores['ih_rih_score'].append(ih_rih_num*100)
        scores['fv_rih_score'].append(fv_rih_num*100)
        scores['rih_ih_score'].append(rih_ih_num*100)
        scores['ih_ih_score'].append(ih_ih_num*100)
        scores['fv_ih_score'].append(fv_ih_num*100)
        scores['rih_fv_score'].append(rih_fv_num*100)
        scores['ih_fv_score'].append(ih_fv_num*100)
        scores['fv_fv_score'].append(fv_fv_num*100)
        
        if 'pythia' in model:
            marker = 'o'
        elif 'gpt2' in model:
            marker = 'x'
        else:
            marker = 's'
        scores['marker'].append(marker)

        x_values.append(x_axis[model])

    if x_label == 'num_params':
        ax.set_xscale('log')

    if head == "induction":
        # ax.plot(x_values, scores['rih_rih_score'], '-', color=colors[2], label='induction')
        # ax.plot(x_values, scores['ih_rih_score'], 'x-', color=colors[3], label='induction')
        ax.plot(x_values, scores['fv_rih_score'], '', color=colors[3], label='fv')
    elif head == "induction":
        ax.plot(x_values, scores['rih_ih_score'], 'x-', color=colors[0], label='induction')
        # ax.plot(x_values, scores['ih_ih_score'], '-', color=colors[3], label='induction')
        ax.plot(x_values, scores['fv_ih_score'], 'x-', color=colors[0], label='fv')
    elif head == "fv":
        ax.plot(x_values, scores['rih_fv_score'], 'x-', color=colors[0], label='induction')
        # ax.plot(x_values, scores['ih_fv_score'], 'x-', color=colors[3], label='induction')
        # ax.plot(x_values, scores['fv_fv_score'], '-', color=colors[4], label='fv')


    # Getting handles and labels
    ax2 = plt.gca()
    handles, labels = ax2.get_legend_handles_labels()

    # by_label = dict(zip(labels, handles))
    # ax.legend(by_label.values(), by_label.keys())

plt.figure()

fig, axs = plt.subplots(1, 2, figsize=(10, 5))

for ax, head in zip(axs.flat, ['induction', 'fv']):
    plot_over_threshold(pythia_models, head, ax)

plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.show()

In [ ]:
import random

def plot_overlap(models,  ax):
    x_values = []
    scores = []
    random_scores = []


    ax.set_xlabel('Model size')
    ax.set_ylabel('Overlap (%)')

    for model in models:
        n_heads = 0.05 * num_heads[model]
        n_heads = np.ceil(n_heads).astype(int)

        ih_score = get_scores(model, "induction")
        fv_score = get_scores(model, "fv")
        top_ih = top_heads(ih_score)[:n_heads]
        top_fv = top_heads(fv_score)[:n_heads]

        x_values.append(num_params[model])
        # scores.append(len(set(top_ih) & set(top_fv)) / len(set(top_ih) | set(top_fv)))
        scores.append(len(set(top_ih) & set(top_fv)) / n_heads * 100)
        random_score = np.mean([len(set(top_ih) & set(random.sample(ih_score.keys(), n_heads))) / n_heads * 100 for _ in range(100)])
        random_scores.append(random_score)
    ax.plot(x_values, random_scores, '--', color='grey', label='Induction & random')
    ax.plot(x_values, scores, '-', color=colors[4], label='Induction & FV')
    ax.set_xscale('log')

In [ ]:
plt.figure()
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))
method = "mean"

x_label = 'num_params'
if x_label == 'num_params':
    # x_values = [num_params[model] for model in models]
    x_axis = num_params
else:
    # x_values = [num_heads[model] for model in models]
    x_axis = num_heads
models = sorted(x_axis.keys(), key=lambda k: x_axis[k], reverse=False)

def plot_percentiles(models, head, ax): 

    x_values = []

    scores = defaultdict(list)

    # ax.set_title(f'{method} {head} scores for top 10% of heads')
    ax.set_xlabel('Model size')
    ax.set_ylabel(f"{head.replace('induction', 'Induction').replace('fv', 'FV')} %ile")
    for model in models:
        n_heads = 0.02 * num_heads[model]
        n_heads = np.ceil(n_heads).astype(int)

        ih_score = get_scores(model, "induction")
        rih_score = get_scores(model, "induction")
        fv_score = get_scores(model, "fv")

        top_ih = top_heads(ih_score)[:n_heads]
        top_rih = top_heads(rih_score)[:n_heads]
        top_fv = top_heads(fv_score)[:n_heads]

        if method == "mean":
            ih_ih_score = np.mean([ih_score[h] for h in top_ih])
            ih_rih_score = np.mean([rih_score[h] for h in top_ih])
            ih_fv_score = np.mean([fv_score[h] for h in top_ih])

            rih_ih_score = np.mean([ih_score[h] for h in top_rih])
            rih_rih_score = np.mean([rih_score[h] for h in top_rih])
            rih_fv_score = np.mean([fv_score[h] for h in top_rih])

            fv_ih_score = np.mean([ih_score[h] for h in top_fv])
            fv_rih_score = np.mean([rih_score[h] for h in top_fv])
            fv_fv_score = np.mean([fv_score[h] for h in top_fv])
        elif method == "min":
            bottom_20 = max(1, int(0.2 * n_heads))
            print(n_heads, bottom_20)
            ih_ih_score = np.mean(sorted([ih_score[h] for h in top_ih[:bottom_20]]))
            ih_rih_score = np.mean(sorted([rih_score[h] for h in top_ih[:bottom_20]]))
            ih_fv_score = np.mean(sorted([fv_score[h] for h in top_ih[:bottom_20]]))

            rih_ih_score = np.mean(sorted([ih_score[h] for h in top_rih[:bottom_20]]))
            rih_rih_score = np.mean(sorted([rih_score[h] for h in top_rih[:bottom_20]]))
            rih_fv_score = np.mean(sorted([fv_score[h] for h in top_rih[:bottom_20]]))

            fv_ih_score = np.mean(sorted([ih_score[h] for h in top_fv[:bottom_20]]))
            fv_rih_score = np.mean(sorted([rih_score[h] for h in top_fv[:bottom_20]]))
            fv_fv_score = np.mean(sorted([fv_score[h] for h in top_fv[:bottom_20]]))
        else:
            ih_ih_score = np.max([ih_score[h] for h in top_ih])
            ih_rih_score = np.max([rih_score[h] for h in top_ih])
            ih_fv_score = np.max([fv_score[h] for h in top_ih])

            rih_ih_score = np.max([ih_score[h] for h in top_rih])
            rih_rih_score = np.max([rih_score[h] for h in top_rih])
            rih_fv_score = np.max([fv_score[h] for h in top_rih])

            fv_ih_score = np.max([ih_score[h] for h in top_fv])
            fv_rih_score = np.max([rih_score[h] for h in top_fv])
            fv_fv_score = np.max([fv_score[h] for h in top_fv])

        ih_scores = sorted(list(ih_score.values()))
        rih_scores = sorted(list(rih_score.values()))
        fv_scores = sorted(list(fv_score.values()))

        scores['rih_rih_score'].append(percentileofscore(rih_scores, rih_rih_score))
        scores['ih_rih_score'].append(percentileofscore(rih_scores, ih_rih_score))
        scores['fv_rih_score'].append(percentileofscore(rih_scores, fv_rih_score))
        scores['rih_ih_score'].append(percentileofscore(ih_scores, rih_ih_score))
        scores['ih_ih_score'].append(percentileofscore(ih_scores, ih_ih_score))
        scores['fv_ih_score'].append(percentileofscore(ih_scores, fv_ih_score))
        scores['rih_fv_score'].append(percentileofscore(fv_scores, rih_fv_score))
        scores['ih_fv_score'].append(percentileofscore(fv_scores, ih_fv_score))
        scores['fv_fv_score'].append(percentileofscore(fv_scores, fv_fv_score))
        
        if 'pythia' in model:
            marker = 'o'
        elif 'gpt2' in model:
            marker = 'x'
        else:
            marker = 's'
        scores['marker'].append(marker)

        x_values.append(x_axis[model])

    if x_label == 'num_params':
        ax.set_xscale('log')

    if head == "induction":
        # ax.plot(x_values, scores['rih_rih_score'], '-', color=colors[2], label='induction')
        # ax.plot(x_values, scores['ih_rih_score'], 'x-', color=colors[3], label='induction')
        ax.plot(x_values, scores['fv_rih_score'], '-', color=colors[3], label='fv')
    elif head == "induction":
        ax.plot(x_values, scores['rih_ih_score'], '-', color=colors[2], label='induction')
        # ax.plot(x_values, scores['ih_ih_score'], '-', color=colors[3], label='induction')
        ax.plot(x_values, scores['fv_ih_score'], '-', color=colors[4], label='fv')
    elif head == "fv":
        ax.plot(x_values, scores['rih_fv_score'], '-', color=colors[2], label='induction')
        # ax.plot(x_values, scores['ih_fv_score'], 'x-', color=colors[3], label='induction')
        # ax.plot(x_values, scores['fv_fv_score'], '-', color=colors[4], label='fv')

    # ax.set_xticks(ticks=x_values, labels=model_names)

    # Getting handles and labels
    ax2 = plt.gca()
    handles, labels = ax2.get_legend_handles_labels()

    # by_label = dict(zip(labels, handles))
    # ax.legend(by_label.values(), by_label.keys())

plt.figure()

fig, axs = plt.subplots(1, 3, figsize=(12, 3))

for ax, head in zip(axs.flat, ['overlap', 'induction', 'fv']):
    if head == 'overlap':
        head = 'induction'
        plot_overlap(models, ax)
    elif head == 'overlap1':
        head = 'induction'
        plot_over_threshold(models, head, ax, 98, 0.02)
    else:
        plot_percentiles(models, head, ax)

plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/head_percentiles.svg, transparent=True)')
plt.show()

In [ ]:
threshold = False
def plot_head_scatter(model, induction_threshold, fv_threshold, ax):
    induction_scores_path = f"./outputs/heads/{model}/induction.pkl"
    fv_scores_path = f"./outputs/heads/{model}/fv.pkl"
    with open(induction_scores_path, 'rb') as f:
        induction_scores = pickle.load(f)
    # top_induction = heads_over_threshold(induction_scores, induction_threshold)
    with open(fv_scores_path, 'rb') as f:
        fv_scores = pickle.load(f)
    # top_fv = heads_over_threshold(fv_scores, fv_threshold)
    if threshold:
        top_induction = heads_over_threshold(induction_scores, induction_threshold)
        top_fv = heads_over_threshold(fv_scores, fv_threshold)
    else:
        n_top_heads = max(2, 0.02 * num_heads[model])
        n_top_heads = int(np.ceil(n_top_heads))
        top_induction = top_heads(induction_scores)[:n_top_heads]
        top_fv = top_heads(fv_scores)[:n_top_heads]
    
    heads = list(set(top_induction) & set(top_fv))


    induction = [induction_scores[head] for head in top_induction]
    fv = [fv_scores[head] for head in top_induction]
    # get spearman correlation
    corr, r = spearmanr(induction, fv)

    ax.scatter(list(induction_scores.values()), list(fv_scores.values()), color=colors[0], marker='o')
    ax.scatter(induction, fv, color=colors[2], label='Induction heads')

    induction = [induction_scores[head] for head in top_fv]
    fv = [fv_scores[head] for head in top_fv]
    ax.scatter(induction, fv, color=colors[3], label='FV heads')

    induction = [induction_scores[head] for head in heads]
    fv = [fv_scores[head] for head in heads]
    ax.scatter(induction, fv, color='purple', label='Induction-FV heads')
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    ax.set_xlabel('Induction score')
    ax.set_ylabel('FV score')
    # ax.set_title(f'{model} corr={corr:.2f}, r={r:.2f}')
    ax.set_title(longnames[model])
    # plt.show()
    handles = ax.get_legend_handles_labels()[0]
    labels = ax.get_legend_handles_labels()[1]
    return handles, labels


fv_threshold = 0
induction_threshold = 0
# fv_threshold = -np.inf
# induction_threshold = -np.inf
fig, axs = plt.subplots(4, 3, figsize=(15, 20))

for ax, model in zip(axs.flat, models):
    try:
        handles, labels = plot_head_scatter(model, induction_threshold, fv_threshold, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3)

plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/corr.pdf', bbox_inches='tight')
plt.show()


In [ ]:
longnames

In [ ]:
sub_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']

num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = True
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        # tasks = ['dictionary']
        fv = 'fv'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_10shot = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    # ax.set_title(longnames[model])
    ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('Accuracy')

    outputs_dir = f"./icml_outputs/scores/{model}"
    non_excl_outputs_dir = f"./icml_outputs/scores/{model.replace('_excl', '')}_zero_shot"
    if excl:
        outputs_dir += "_excltop2_zero_shot"
    for num_ablate in num_ablates:
        bases = []
        base_10shot = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    bases.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"./icml_outputs/scores/pythia-6.9b/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    base_10shot.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    fvs.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    randoms.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    rand_inductions.append(np.mean(scores))
            except:
                pass
        # print(len(inductions))
        bases = np.mean([float(x) for x in bases])
        base10 = np.mean([float(x) for x in base_10shot])
        inductions = np.mean([float(x) for x in inductions])
        fvs = np.mean([float(x) for x in fvs])
        randoms = np.mean([float(x) for x in randoms])
        rand_inductions = np.mean([float(x) for x in rand_inductions])
    
        all_bases[model].append(bases)
        all_10shot[model].append(base10)
        all_inductions[model].append(inductions)
        all_fvs[model].append(fvs)
        all_randoms[model].append(randoms)
        all_rand_inductions[model].append(rand_inductions)

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
    ax.plot(p_ablates, all_10shot[model], label=f'Clean (10-shot)', color='red')
    ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
    # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
    ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2])
    ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])

  
plt.figure()
fig, axs = plt.subplots(1, 1, figsize=(6, 4))
plot_task_ablations('pythia-6.9b', axs)

# for ax, model in zip(axs.flat, sub_models):
#     try:
#         plot_task_ablations(model, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.legend()
plt.title('Pythia-6.9B (zero-shot)')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/abl_zero_shot.pdf')
plt.show()

In [ ]:
def get_heads_to_ablate(head_path, abl_head_name, num_ablate, exclude_other_heads):
    other_head_name = 'induction' if 'fv' in abl_head_name else 'fv_0' # other head is fv if ablating induction/rand_induction, and vice versa

    if num_ablate == 0:
        return []
    
    if abl_head_name == 'random':
        with open(f'{head_path}/induction.pkl', 'rb') as f: # load induction heads just to get the keys
            tmp_heads = pickle.load(f)
        all_heads = {k: random.random() for k in tmp_heads.keys()}
    elif abl_head_name == 'copy-induction':
        with open(f'{head_path}/induction.pkl', 'rb') as f:
            induction_heads = pickle.load(f)
        with open(f'{head_path}/copying.pkl', 'rb') as f:
            copying_heads = pickle.load(f)

        induction_weight = 1/3
        copying_weight = 2/3 # copy score range is 0 to 0.5
        all_heads = {k: induction_weight * induction_heads[k] + copying_weight * copying_heads[k] for k in induction_heads.keys()}
        if exclude_other_heads:
            with open(f'{head_path}/{other_head_name}.pkl', 'rb') as f:
                other_heads = pickle.load(f)
    else:
        with open(f'{head_path}/{abl_head_name}.pkl', 'rb') as f:
            all_heads = pickle.load(f)
        if exclude_other_heads:
            with open(f'{head_path}/{other_head_name}.pkl', 'rb') as f:
                other_heads = pickle.load(f)

    all_heads = sorted(all_heads.keys(), key=lambda k: all_heads[k], reverse=True)

    if num_ablate < 1:
        num_ablate = int(num_ablate * len(all_heads))
    if num_ablate == 0:
        num_ablate = 1

    if exclude_other_heads:
        num_exclude = int(0.02 * len(all_heads)) #min(num_ablate, int(0.02 * len(all_heads)))
        other_heads = sorted(other_heads.keys(), key=lambda k: other_heads[k], reverse=True)
        all_heads = [h for h in all_heads if h not in other_heads[:int(num_exclude)]]

    heads_to_ablate = list(
        map(lambda x: list(map(int, x.split("."))), all_heads[:int(num_ablate)])
    )
    return heads_to_ablate

In [ ]:
def plot_icl_scores(model, ax):
    num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
    models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b','pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']
    heads = ['induction', 'fv']
    clean_dir = f"./outputs/icl-scores/{model}"
    outputs_dir = f"./outputs/icl-scores/{model}_excl"
    # ax.set_title(longnames[model])
    ax.set_xlabel('Ablation (%)')
    if model == sub_models[0]:
        ax.set_ylabel('TL difference')

    all_inductions = []
    all_fvs = []
    all_randoms = []
    
    with open(f"{clean_dir}/random_0.0_score.txt", "r") as f:
        clean_score = float(f.readlines()[0].strip())

    ih_score = []
    fv_score = []
    random_score = []
    rand_ih_score = []
    clean_scores = []
    copy_score = []
    copy_ih_score = []
    x_ih = []
    x_fv = []
    x_random = []
    x_rand_ih = []
    x_clean = []
    x_copy = []
    x_copy_ih = []
    for n in num_ablates:
        clean_scores.append(clean_score)
        x_clean.append(n*100)
        try:
            with open(f"{outputs_dir}/induction_{n}_50.txt", "r") as f:
                loss_50 = float(f.readlines()[0].strip())
            with open(f"{outputs_dir}/induction_{n}_500.txt", "r") as f:
                loss_500 = float(f.readlines()[0].strip())
            ih_score.append(loss_50 - loss_500)
            x_ih.append(n*100)
        except:
            print(f"{model} {n} induction not found")
        try:
            with open(f"{outputs_dir}/fv_{n}_50.txt", "r") as f:
                loss_50 = float(f.readlines()[0].strip())
            with open(f"{outputs_dir}/fv_{n}_500.txt", "r") as f:
                loss_500 = float(f.readlines()[0].strip())
            fv_score.append(loss_50 - loss_500)
            x_fv.append(n*100)
        except:
            print(f"{model} {n} fv not found")
        try:
            with open(f"{clean_dir}/random_{n}_50.txt", "r") as f:
                loss_50 = float(f.readlines()[0].strip())
            with open(f"{clean_dir}/random_{n}_500.txt", "r") as f:
                loss_500 = float(f.readlines()[0].strip())
            random_score.append(loss_50 - loss_500)
            x_random.append(n*100)
        except:
            print(f"{model} {n} random not found")
        try:
            with open(f"{outputs_dir}/induction_{n}_50.txt", "r") as f:
                loss_50 = float(f.readlines()[0].strip())
            with open(f"{outputs_dir}/induction_{n}_500.txt", "r") as f:
                loss_500 = float(f.readlines()[0].strip())
            rand_ih_score.append(loss_50 - loss_500)
            x_rand_ih.append(n*100)
        except:
            print(f"{model} {n} random induction not found")
        # try:
        #     with open(f"{outputs_dir}/copying_{n}_50.txt", "r") as f:
        #         loss_50 = float(f.readlines()[0].strip())
        #     with open(f"{outputs_dir}/copying_{n}_500.txt", "r") as f:
        #         loss_500 = float(f.readlines()[0].strip())
        #     copy_score.append(loss_50 - loss_500)
        #     x_copy.append(n)
        # except:
        #     print(f"{model} {n} copy not found")
        # try:
        #     with open(f"{outputs_dir}/copy-induction_{n}_50.txt", "r") as f:
        #         loss_50 = float(f.readlines()[0].strip())
        #     with open(f"{outputs_dir}/copy-induction_{n}_500.txt", "r") as f:
        #         loss_500 = float(f.readlines()[0].strip())
        #     copy_ih_score.append(loss_50 - loss_500)
        #     x_copy_ih.append(n)
        # except:
        #     print(f"{model} {n} copy induction not found")
    ax.plot(x_clean, clean_scores, label='Clean', color=colors[5])
    ax.plot(x_random, random_score, label='Random', color=colors[0])
    # ax.plot(x_ih, ih_score, label='Induction (data)',  marker='o', color=colors[4])
    ax.plot(x_rand_ih, rand_ih_score, label='Induction', marker='o', color=colors[2], linestyle='--')
    ax.plot(x_fv, fv_score, label='FV', marker='x', color=colors[3], linestyle='--')
    # ax.plot(x_copy, copy_score, label='copy', marker='x')
    # ax.plot(x_copy_ih, copy_ih_score, label='copy induction', marker='x')
    # if '6.9' in model:
    #     ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    handles = ax.get_legend_handles_labels()[0]
    labels = ax.get_legend_handles_labels()[1]
    return handles, labels



# plt.figure()
# fig, axs = plt.subplots(4, 3, figsize=(15, 12))
# # fig, axs = plt.subplots(1, 3, figsize=(15, 3))

# models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
# sub_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']
# for ax, model in zip(axs.flat, models):
#     try:
#         handles, labels = plot_icl_scores(model, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.1), ncol=4)
# plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
# plt.savefig('plots/icl_scores_all.pdf', bbox_inches='tight')
# plt.show()
  

In [ ]:
from matplotlib.lines import Line2D
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = False
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    all_copyings = defaultdict(list)
    all_copy_inductions = defaultdict(list)
    all_overlaps = defaultdict(list)
    if not excl:
        ax.set_title(longnames[model])

    # ax.set_xlabel('Ablation (%)')
    if model == sub_models[0]:
        ax.set_ylabel('ICL Accuracy')

    head_path = f"./outputs/heads/{model}"
    


    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    if excl:
        outputs_dir += "_excltop2"
    for num_ablate in num_ablates:
        bases = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        copyings = []
        copy_inductions = []


        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    bases.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    fvs.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    randoms.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    rand_inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copying_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copyings.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copy-induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copy_inductions.append(np.mean(scores))
            except:
                pass
        # print(len(inductions))
        bases = np.mean([float(x) for x in bases])
        inductions = np.mean([float(x) for x in inductions])
        fvs = np.mean([float(x) for x in fvs])
        randoms = np.mean([float(x) for x in randoms])
        rand_inductions = np.mean([float(x) for x in rand_inductions])
        copyings = np.mean([float(x) for x in copyings])
        copy_inductions = np.mean([float(x) for x in copy_inductions])

        all_bases[model].append(bases)
        all_inductions[model].append(inductions)
        all_fvs[model].append(fvs)
        all_randoms[model].append(randoms)
        all_rand_inductions[model].append(rand_inductions)
        all_copyings[model].append(copyings)
        all_copy_inductions[model].append(copy_inductions)
        

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
    # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
    ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
    if excl:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2], linestyle='--')
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3], linestyle='--')
    else:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(3, 3, figsize=(15, 8))
# plot_task_ablations('pythia-6.9b', axs)

for ax, (excl, model) in zip(axs.flat, itertools.product([False, True, 'hi'], sub_models)):
    try:
        if excl == False:
            handles, labels = plot_task_ablations(model, ax)
        elif excl == True:
            plot_task_ablations(model, ax)
        else:
            plot_icl_scores(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
handles.append(custom_handle)
labels.append('Ablation with exclusion')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=5)
plt.savefig('plots/abl_6.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = False
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    all_copyings = defaultdict(list)
    all_copy_inductions = defaultdict(list)
    all_overlaps = defaultdict(list)
    if not excl:
        ax.set_title(longnames[model])
    else:
        ax.set_title(f'{longnames[model]} w/ exclusion')
    ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('ICL Accuracy')

    head_path = f"./outputs/heads/{model}"
    


    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    if excl:
        outputs_dir += "_excltop2"
    for num_ablate in num_ablates:
        bases = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        copyings = []
        copy_inductions = []


        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    bases.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    fvs.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    randoms.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    rand_inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copying_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copyings.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copy-induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copy_inductions.append(np.mean(scores))
            except:
                pass
        # print(len(inductions))
        bases = np.mean([float(x) for x in bases])
        inductions = np.mean([float(x) for x in inductions])
        fvs = np.mean([float(x) for x in fvs])
        randoms = np.mean([float(x) for x in randoms])
        rand_inductions = np.mean([float(x) for x in rand_inductions])
        copyings = np.mean([float(x) for x in copyings])
        copy_inductions = np.mean([float(x) for x in copy_inductions])

        induction_heads = get_heads_to_ablate(head_path, 'induction', num_ablate, excl)
        fv_heads = get_heads_to_ablate(head_path, 'fv', num_ablate, excl)
    
        all_bases[model].append(bases)
        all_inductions[model].append(inductions)
        all_fvs[model].append(fvs)
        all_randoms[model].append(randoms)
        all_rand_inductions[model].append(rand_inductions)
        all_copyings[model].append(copyings)
        all_copy_inductions[model].append(copy_inductions)
        

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
    # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
    ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
    if excl:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2], linestyle='--')
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3], linestyle='--')
    else:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(6, 4, figsize=(15, 18))
# plot_task_ablations('pythia-6.9b', axs)

for ax, (model, excl) in zip(axs.flat, itertools.product(models, [False, True])):
    try:
        if not excl:
            handles, labels = plot_task_ablations(model, ax)
        else:
            plot_task_ablations(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
handles.append(custom_handle)
labels.append('Ablation with exclusion')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=5)
plt.savefig('plots/abl_all.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = False
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        # tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_overlaps = defaultdict(list)
    if not excl:
        ax.set_title(longnames[model])
    else:
        ax.set_title(f'{longnames[model]} w/ exclusion')
    ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('Overlap (%)')

    head_path = f"./outputs/heads/{model}"
    


    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    if excl:
        outputs_dir += "_excltop2"
    for num_ablate in num_ablates:


        induction_heads = get_heads_to_ablate(head_path, 'induction', num_ablate, excl)
        fv_heads = get_heads_to_ablate(head_path, 'fv', num_ablate, excl)
        induction_heads = list(map(tuple, induction_heads))
        fv_heads = list(map(tuple, fv_heads))
        overlap = len(set(induction_heads) & set(fv_heads)) / len(induction_heads) * 100
    

        all_overlaps[model].append(overlap)

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_overlaps[model], label=f'Overlap', color=colors[4])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(3, 4, figsize=(15, 9))
# plot_task_ablations('pythia-6.9b', axs)

for ax, model in zip(axs.flat, models):
    # try:
    if not excl:
        handles, labels = plot_task_ablations(model, ax)
    else:
        plot_task_ablations(model, ax)
    # except Exception as e:
    #     print(e)
    #     print(model)
    #     ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/abl_overlap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
get_scores('pythia-6.9b', 'induction')

In [ ]:
scores =get_scores(model, score)
all_heads = list(scores.keys())
all_heads = list(map(lambda x: tuple(list(map(int, x.split(".")))), all_heads))
all_heads

In [ ]:
from matplotlib.lines import Line2D
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)

val_only = True
def plot_preserved_heads(model, ax, score, percentile=False):

    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    scores_of_fv = defaultdict(list)
    scores_of_induction = defaultdict(list)

    if not excl:
        ax.set_title(longnames[model])
    else:
        ax.set_title(f'{longnames[model]} w/ exclusion')
    ax.set_xlabel('Ablation (%)')
    if 'induction' in score:
        ylabel = 'Induction score'
    elif 'fv' in score:
        ylabel = 'FV score'
    
    if percentile:
        ylabel = ylabel.replace('score', '%ile')
    ax.set_ylabel(ylabel)

    head_path = f"./outputs/heads/{model}"
    
    scores = get_scores(model, score)

    all_scores = list(scores.values())
    all_heads = list(scores.keys())
    all_heads = list(map(lambda x: tuple(list(map(int, x.split(".")))), all_heads))


    for num_ablate in num_ablates:


        induction_heads = get_heads_to_ablate(head_path, 'induction', num_ablate, excl)
        fv_heads = get_heads_to_ablate(head_path, 'fv', num_ablate, excl)
        induction_heads = list(map(tuple, induction_heads))
        fv_heads = list(map(tuple, fv_heads))
        induction_heads = [head for head in all_heads if head not in induction_heads]
        fv_heads = [head for head in all_heads if head not in fv_heads]


        
        
    
        score_of_fv = np.mean([scores[f'{head[0]}.{head[1]}'] for head in fv_heads])
        score_of_induction = np.mean([scores[f'{head[0]}.{head[1]}'] for head in induction_heads])
        if percentile:
            score_of_fv = percentileofscore(all_scores, score_of_fv)
            score_of_induction = percentileofscore(all_scores, score_of_induction)

        scores_of_fv[model].append(score_of_fv)
        scores_of_induction[model].append(score_of_induction)


    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, scores_of_induction[model], label=f'Induction', color=colors[2])
    ax.plot(p_ablates, scores_of_fv[model], label=f'FV', color=colors[3])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(3, 4, figsize=(15, 9))
# plot_task_ablations('pythia-6.9b', axs)
excl = True
score = 'fv'
percentile = True

for ax, model in zip(axs.flat, models):
    # try:
    if not excl:
        handles, labels = plot_preserved_heads(model, ax, score, percentile)
    else:
        plot_preserved_heads(model, ax, score, percentile)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=5)
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/abl_fv_percentiles_excl.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D

num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = True
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    all_copyings = defaultdict(list)
    all_copy_inductions = defaultdict(list)
    if excl:
        ax.set_title(longnames[model])
    if excl:
        ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('Few-shot ICL Accuracy')
    ax.set_xlabel('Ablation (%)')

    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    if excl:
        outputs_dir += "_excltop2"
    for num_ablate in num_ablates:
        bases = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        copyings = []
        copy_inductions = []

        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    bases.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    fvs.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    randoms.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    rand_inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copying_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copyings.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_copy-induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    copy_inductions.append(np.mean(scores))
            except:
                pass
        # print(len(inductions))
        bases = np.mean([float(x) for x in bases])
        inductions = np.mean([float(x) for x in inductions])
        fvs = np.mean([float(x) for x in fvs])
        randoms = np.mean([float(x) for x in randoms])
        rand_inductions = np.mean([float(x) for x in rand_inductions])
        copyings = np.mean([float(x) for x in copyings])
        copy_inductions = np.mean([float(x) for x in copy_inductions])
    
        all_bases[model].append(bases)
        all_inductions[model].append(inductions)
        all_fvs[model].append(fvs)
        all_randoms[model].append(randoms)
        all_rand_inductions[model].append(rand_inductions)
        all_copyings[model].append(copyings)
        all_copy_inductions[model].append(copy_inductions)

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
    # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
    ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
    if not excl:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2], linestyle='--')
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3], linestyle='--')
    else:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(1,1, figsize=(8, 4))
handles, labels =  plot_task_ablations('pythia-6.9b', axs)

# for ax, (excl, model) in zip(axs.flat, itertools.product([False, True], sub_models)):
#     try:
#         if not excl:
#             handles, labels = plot_task_ablations(model, ax)
#         else:
#             plot_task_ablations(model, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
# handles.append(custom_handle)
# labels.append('Ablation with exclusion')
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.05), ncol=2)
fig.legend(handles, labels, loc='right',ncol=1, fontsize='small', bbox_to_anchor=(0.95, 0.45))

plt.savefig('plots/abl_1_trpt.svg', transparent=True, bbox_inches='tight', )
plt.show()

In [ ]:
num_ablates

In [ ]:
from matplotlib.lines import Line2D

num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = True
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    bases = []
    randoms = [0.744, 0.723, 0.716, 0.719, 0.642, 0.662, 0.615]
    inductions = [0.753, 0.731, 0.726, 0.729, 0.623, 0.514, 0.489]
    fvs = [0.739, 0.728, 0.610, 0.382, 0.294, 0.301, 0.289]
    if excl:
        ax.set_title(longnames[model])
    if excl:
        ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('ICL Accuracy')
    ax.set_xlabel('Ablation (%)')

    outputs_dir = f"./icml_outputs/scores/{model}"
    for num_ablate in num_ablates:
        base_tasks = []
        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{outputs_dir}/{task}_induction_0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    base_tasks.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            
        bases.append(np.mean(base_tasks))
       

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, bases, label=f'Clean', color=colors[5])
    ax.plot(p_ablates, randoms, label=f'Random', marker='', color=colors[0])
    if not excl:
        ax.plot(p_ablates, inductions, label=f'Induction', marker='o', color=colors[2], linestyle='--')
        ax.plot(p_ablates, fvs, label=f'FV', marker='x', color=colors[3], linestyle='--')
    else:
        ax.plot(p_ablates, inductions, label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, fvs, label=f'FV', marker='x', color=colors[3])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(1,1, figsize=(8, 4))
handles, labels =  plot_task_ablations('Meta-Llama-3-8B', axs)

# for ax, (excl, model) in zip(axs.flat, itertools.product([False, True], sub_models)):
#     try:
#         if not excl:
#             handles, labels = plot_task_ablations(model, ax)
#         else:
#             plot_task_ablations(model, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
# handles.append(custom_handle)
# labels.append('Ablation with exclusion')
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.05), ncol=2)
fig.legend(handles, labels, loc='right',ncol=1, fontsize='small', bbox_to_anchor=(0.95, 0.45))

plt.savefig('plots/abl_llama3.pdf', bbox_inches='tight', )
plt.show()

In [ ]:
from matplotlib.lines import Line2D

num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = True
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    bases = []
    randoms = [0.755, 0.753, 0.716, 0.729, 0.672, 0.632, 0.625]
    inductions = [0.751, 0.741, 0.721, 0.701, 0.653, 0.532, 0.501]
    fvs = [0.714, 0.658, 0.610, 0.523, 0.318, 0.295, 0.302]
    if excl:
        ax.set_title(longnames[model])
    if excl:
        ax.set_xlabel('Ablation (%)')
    ax.set_ylabel('ICL Accuracy')
    ax.set_xlabel('Ablation (%)')

    outputs_dir = f"./icml_outputs/scores/{model}"
    for num_ablate in num_ablates:
        base_tasks = []
        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{outputs_dir}/{task}_induction_0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    base_tasks.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            
        bases.append(np.mean(base_tasks))
       

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, bases, label=f'Clean', color=colors[5])
    ax.plot(p_ablates, randoms, label=f'Random', marker='', color=colors[0])
    if not excl:
        ax.plot(p_ablates, inductions, label=f'Induction', marker='o', color=colors[2], linestyle='--')
        ax.plot(p_ablates, fvs, label=f'FV', marker='x', color=colors[3], linestyle='--')
    else:
        ax.plot(p_ablates, inductions, label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, fvs, label=f'FV', marker='x', color=colors[3])
    # ax.plot(p_ablates, all_copyings[model], label=f'Copy', marker='o', color=colors[4])
    # ax.plot(p_ablates, all_copy_inductions[model], label=f'Copy Induction', marker='o', color=colors[1])
    return ax.get_legend_handles_labels()

    
    
  
plt.figure()
fig, axs = plt.subplots(1,1, figsize=(8, 4))
handles, labels =  plot_task_ablations('Qwen2.5-7B', axs)

# for ax, (excl, model) in zip(axs.flat, itertools.product([False, True], sub_models)):
#     try:
#         if not excl:
#             handles, labels = plot_task_ablations(model, ax)
#         else:
#             plot_task_ablations(model, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
# handles.append(custom_handle)
# labels.append('Ablation with exclusion')
# fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.05), ncol=2)
fig.legend(handles, labels, loc='right',ncol=1, fontsize='small', bbox_to_anchor=(0.95, 0.45))

plt.savefig('plots/abl_qwen.pdf', bbox_inches='tight', )
plt.show()

In [ ]:
num_ablates = [0.01, 0.03, 0.05, 0.07, 0.09, 0.15, 0.2]
models = sorted(num_params.keys(), key=lambda k: num_params[k], reverse=False)
excl = False
val_only = True
def plot_task_ablations(model, ax):
    outputs_dir = f"./outputs/scores/pythia-70m"
    if val_only:
        # tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        # tasks = [x for x in tasks if x not in test_data]
        # tasks = list(map(lambda x: x.replace("_val",""), new_tasks))
        tasks = test_data #list(set(test_data) & set([x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]))
        fv = 'fv_0'
    else:
        tasks = [x.replace(f'_random_0.0.txt', '').replace(outputs_dir+'/', '') for x in glob.glob(f'{outputs_dir}/*_random_0.0.txt')]
        fv = 'fv'
    # For each model, plot line plots of scores on y axis and num_ablte on x axis with different line for each type of score averaged across tasks


    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    all_bases = defaultdict(list)
    all_inductions = defaultdict(list)
    all_fvs = defaultdict(list)
    all_randoms = defaultdict(list)
    all_rand_inductions = defaultdict(list)
    all_copy = defaultdict(list)
    all_copy_ih = defaultdict(list)
    if not excl:
        ax.set_title(longnames[model])
    if excl:
        ax.set_title(f'{longnames[model]} (with exclusion)')
    ax.set_xlabel('Ablation (%)')
    if model == 'pythia-410m':
        ax.set_ylabel('Accuracy')

    outputs_dir = f"./outputs/scores/{model}"
    non_excl_outputs_dir = f"./outputs/scores/{model.replace('_excl', '')}"
    copy_outputs_dir = outputs_dir
    if excl:
        outputs_dir += "_excltop2"
        copy_outputs_dir += "_excltop2"
    for num_ablate in num_ablates:
        bases = []
        inductions = []
        rand_inductions = []
        fvs = []
        randoms = []
        copyings = []
        copy_inductions = []
        for task in tasks:
            if 'icl' in task:
                continue
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_0.0.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    bases.append(np.mean(scores))
            except Exception as e:
                print(e)
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    inductions.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_{fv}_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    fvs.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{non_excl_outputs_dir}/{task}_random_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    randoms.append(np.mean(scores))
            except:
                pass
            try:
                with open(f"{outputs_dir}/{task}_induction_{num_ablate}.txt", "r") as f:
                    scores = list(map(float, f.readlines()))
                    rand_inductions.append(np.mean(scores))
            except:
                pass
            # try:
            #     with open(f"{copy_outputs_dir}/{task}_copying_{num_ablate}.txt", "r") as f:
            #         scores = list(map(float, f.readlines()))
            #         copyings.append(np.mean(scores))
            # except:
            #     pass
            # try:
            #     with open(f"{copy_outputs_dir}/{task}_copy-induction_{num_ablate}.txt", "r") as f:
            #         scores = list(map(float, f.readlines()))
            #         copy_inductions.append(np.mean(scores))
            # except:
            #     pass
        # print(len(inductions))
        bases = np.mean([float(x) for x in bases])
        inductions = np.mean([float(x) for x in inductions])
        fvs = np.mean([float(x) for x in fvs])
        randoms = np.mean([float(x) for x in randoms])
        rand_inductions = np.mean([float(x) for x in rand_inductions])
        # copyings = np.mean([float(x) for x in copyings])
        # copy_inductions = np.mean([float(x) for x in copy_inductions])
    
        all_bases[model].append(bases)
        all_inductions[model].append(inductions)
        all_fvs[model].append(fvs)
        all_randoms[model].append(randoms)
        all_rand_inductions[model].append(rand_inductions)
        # all_copy[model].append(copyings)
        # all_copy_ih[model].append(copy_inductions)

    p_ablates = [n * 100 for n in num_ablates]
    ax.plot(p_ablates, all_bases[model], label=f'Clean', color=colors[5])
    # ax.plot(num_ablates, all_inductions[model], label=f'induction', marker='o', color=colors[0])
    if excl:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction (random)', marker='o', linestyle=':', color=colors[4])
        ax.plot(p_ablates, all_inductions[model], label=f'Induction', marker='o', linestyle=':', color=colors[2])
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', linestyle=':', color=colors[3])
    else:
        ax.plot(p_ablates, all_rand_inductions[model], label=f'Induction (random)', marker='o', color=colors[4])
        ax.plot(p_ablates, all_inductions[model], label=f'Induction', marker='o', color=colors[2])
        ax.plot(p_ablates, all_fvs[model], label=f'FV', marker='x', color=colors[3])
    ax.plot(p_ablates, all_randoms[model], label=f'Random', marker='', color=colors[0])
    # ax.plot(p_ablates, all_copy[model], label=f'Copy', marker='x', color=colors[4])
    # ax.plot(p_ablates, all_copy_ih[model], label=f'Copy Induction', marker='x', linestyle=':', color=colors[4])

    
    
  
plt.figure()
fig, axs = plt.subplots(6, 4, figsize=(20, 15))
# plot_task_ablations('pythia-6.9b', axs)

for ax, (model, excl) in zip(axs.flat, itertools.product(models, [False, True])):
    try:
        plot_task_ablations(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.

plt.legend(fontsize='small', loc='lower right')
plt.savefig('plots/abl_all.pdf')
plt.show()

In [ ]:
models

In [ ]:
print(np.log2(np.array(checkpoints)))

In [ ]:
fv_paper_train = ['product-company_train', 'landmark-country_train', 'sentiment_train', 'antonym_train', 
                    'concept_v_object_5_train', 'concept_v_object_3_train', 'fruit_v_animal_5_train', 'synonym_train', 
                    'lowercase_first_letter_train', 'object_v_concept_5_train', 'choose_middle_of_5_train', 
                    'conll2003_person_train', 'color_v_animal_5_train', 'choose_last_of_3_train', 'adjective_v_verb_3_train', 
                    'verb_v_adjective_5', 'animal_v_object_5_val', 'conll2003_location_train', 'adjective_v_verb_5_train', 
                    'color_v_animal_3_train', 'choose_middle_of_3_train', 'object_v_concept_3_train', 'capitalize_train', 
                    'fruit_v_animal_3_train', 'choose_last_of_5_train', 'choose_first_of_3_train', 
                    'verb_v_adjective_3_train', 'english-spanish_train', 'english-french_train']

fv_paper_test = ['ag_news_val', 'capitalize_first_letter_val', 'conll2003_organization_val', 'national_parks_val', 'park-country_val', 
                    'person-occupation_val', 'animal_v_object_3_val', 'choose_first_of_5_val', 
                    'english-german_val', 'person-instrument_val', 'commonsense_qa_val']

new_tasks = ['capital_index_val',
    'capitalize_second_letter_train', 
    'abstract_clf_val',
    'french-english_val',
    'bind_fruit_val',
    'bind_capital_par_val',
    'bind_capital_val',
    'bind_shape_val']

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]
model = 'pythia-2.8b'

x_label = 'Steps'

cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))


def plot_head_ckpt(model, head, ax):
    scores = defaultdict(list)

    ax.set_title(longnames[model])
    ax.set_xlabel('Steps')
    

    for ckpt in checkpoints:
        if ckpt == 143000:
            head_path = f"./outputs/heads/{model}/{head}.pkl"
            head_score = get_scores(model, head)
        else:
            head_path = f"./outputs/heads/{model}-{ckpt}/{head}.pkl"
            head_score = get_scores(model, head, ckpt)

        # with open(head_path, 'rb') as f:
        #     heads = pickle.load(f)
        heads = head_score
        values = list(heads.values())
        mean = np.mean(values)
        median = np.median(values)
        if 'induction' in head:
            t1, t2, t3 = 0.1, 0.2, 0.3
        else:
            t1, t2, t3 = 1e-5, 2e-5, 3e-5
        gt_02 = sum(np.array(values) > t1)
        gt_05 = sum(np.array(values) > t2)
        gt_07 = sum(np.array(values) > t3)
        total = len(values)

        
        n = 0.02 * num_heads[model]
        n = np.ceil(n).astype(int)
        # top_head_scores = top_heads(head_score)[:n]
        top_head_scores = top_heads(get_scores(model, 'fv'))[:n]
        top_head_mean = np.mean([head_score[h] for h in top_head_scores])

        scores['mean'].append(mean)
        scores['median'].append(median)
        scores['gt_02'].append(gt_02/total)
        scores['gt_05'].append(gt_05/total)
        scores['gt_07'].append(top_head_mean)
        scores['ckpt'].append(ckpt)
        if 'pythia' in model:
            marker = 'o'
        elif 'gpt2' in model:
            marker = 'x'
        else:
            marker = 's'
        scores['marker'].append(marker)
        # ckpt = np.log2(ckpt)
        ax.set_xscale('log', base=2)

        # ax.plot(scores['ckpt'], scores['gt_02'], label=f'>{t1}', marker=marker, color=colors[2])
        # ax.plot(scores['ckpt'], scores['gt_05'], label=f'>{t2}', marker=marker, color=colors[3])
        if ckpt == 143000:
            if head == 'fv':
                ax.plot(scores['ckpt'], scores['gt_07'], label=f'FV', marker='', color=colors[3])
            else:
                ax.plot(scores['ckpt'], scores['gt_07'], label=f'Induction', marker='', color=colors[2])
        else:
            if head == 'fv':
                ax.plot(scores['ckpt'], scores['gt_07'],  marker='', color=colors[3])
            else:
                ax.plot(scores['ckpt'], scores['gt_07'], marker='', color=colors[2])


    x = np.array(scores['ckpt'])
    xp = np.linspace(x.min(), x.max(), 100)
    ax.set_ylim(bottom=0)
    # if 'induction' in head:
    #     ax.set_ylim(top=0.5)
    # else:
    #     ax.set_ylim(top=0.09)
    # set regular y ticks
    y_min, y_max = ax.get_ylim()
    yticks = np.linspace(y_min, y_max, 10)






In [ ]:
head

In [ ]:
from matplotlib.ticker import ScalarFormatter

checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]

plt.figure()
# fig, axs = plt.subplots(3, 3, figsize=(15, 9))
fig, ax = plt.subplots(1,1, figsize=(8, 4))
model = 'pythia-6.9b'

n = 2
i = 0
head = top_heads(get_scores(model, 'fv'))[n]
# for i, head in enumerate(fv_heads[:n]):
induction_scores = []
for ckpt in checkpoints:
    if ckpt == 143000:
        head_score = get_scores(model, 'induction')
    else:
        head_score = get_scores(model, 'induction', ckpt)
    induction_scores.append(head_score[head])

ax.plot(checkpoints, induction_scores, label=f'{head}', marker='', color=colors[2], alpha=1 - (i/(1+n)))
ax.set_title(f'Head {head} in {longnames[model]}')
ax.set_ylabel('Induction score', color=colors[2])
ax.set_xlabel('Training steps')
ax.set_xscale('log', base=2)
ax.set_ylim(bottom=-0.1, top=0.5)
ax2 = ax.twinx()
ax2.set_ylabel('FV score', color=colors[3])
ax2.set_ylim(bottom=-0.004, top=0.017)
# for i, head in enumerate(fv_heads[:n]):
fv_scores = []
for ckpt in checkpoints:
    if ckpt == 143000:
        head_score = get_scores(model, 'fv')
    else:
        head_score = get_scores(model, 'fv', ckpt)
    fv_scores.append(head_score[head])

ax2.plot(checkpoints, fv_scores, label=f'{head}', marker='', color=colors[3], alpha=1 - (i/(1+n)))
# plt.gca().ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax2.set_yticks([0, 0.01, 0.02])

# handles, labels = ax3.get_legend_handles_labels()
# by_label = dict(zip(labels, handles))

# (zip(labels, handles))
# plt.legend(by_label.values(), by_label.keys())
# ax.legend()
# plt.legend()
# plt.legend()
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/fig1.svg', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]
model = 'pythia-2.8b'

x_label = 'Steps'

cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))


def plot_head_ckpt(model, head, ax):
    scores = defaultdict(list)

    ax.set_title(longnames[model])
    ax.set_xlabel('Training steps')
    

    for ckpt in checkpoints:
        if ckpt == 143000:
            head_path = f"./outputs/heads/{model}/{head}.pkl"
            head_score = get_scores(model, head)
        else:
            head_path = f"./outputs/heads/{model}-{ckpt}/{head}.pkl"
            head_score = get_scores(model, head, ckpt)

        with open(head_path, 'rb') as f:
            heads = pickle.load(f)
        values = list(heads.values())
        mean = np.mean(values)
        median = np.median(values)
        if 'induction' in head:
            t1, t2, t3 = 0.1, 0.2, 0.3
        else:
            t1, t2, t3 = 1e-5, 2e-5, 3e-5
        gt_02 = sum(np.array(values) > t1)
        gt_05 = sum(np.array(values) > t2)
        gt_07 = sum(np.array(values) > t3)
        total = len(values)

        
        n = 0.02 * num_heads[model]
        n = np.ceil(n).astype(int)
        top_head_scores = top_heads(head_score)[:n]
        top_head_mean = np.mean([head_score[h] for h in top_head_scores])

        scores['mean'].append(mean)
        scores['median'].append(median)
        scores['gt_02'].append(gt_02/total)
        scores['gt_05'].append(gt_05/total)
        scores['gt_07'].append(top_head_mean)
        scores['ckpt'].append(ckpt)
        if 'pythia' in model:
            marker = 'o'
        elif 'gpt2' in model:
            marker = 'x'
        else:
            marker = 's'
        scores['marker'].append(marker)
        # ckpt = np.log2(ckpt)
        ax.set_xscale('log', base=2)

        # ax.plot(scores['ckpt'], scores['gt_02'], label=f'>{t1}', marker=marker, color=colors[2])
        # ax.plot(scores['ckpt'], scores['gt_05'], label=f'>{t2}', marker=marker, color=colors[3])
        if ckpt == 143000:
            if head == 'fv':
                ax.plot(scores['ckpt'], scores['gt_07'], label=f'FV score', marker='', color=colors[3])
            else:
                ax.plot(scores['ckpt'], scores['gt_07'], label=f'Induction score', marker='', color=colors[2])
        else:
            if head == 'fv':
                ax.plot(scores['ckpt'], scores['gt_07'],  marker='', color=colors[3])
            else:
                ax.plot(scores['ckpt'], scores['gt_07'], marker='', color=colors[2])


    x = np.array(scores['ckpt'])
    xp = np.linspace(x.min(), x.max(), 100)
    ax.set_ylim(bottom=0)
    # if 'induction' in head:
    #     ax.set_ylim(top=0.5)
    # else:
    #     ax.set_ylim(top=0.09)
    # set regular y ticks
    y_min, y_max = ax.get_ylim()
    yticks = np.linspace(y_min, y_max, 10)

In [ ]:

def plot_icl(model, ax):
    tasks = test_data#list(map(lambda x: x.replace("_val",""), new_tasks))
    checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]
    outputs_dir = f"./outputs/scores/"

    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    ax.set_xscale('log')
    ax.set_title(longnames[model])
    ax.set_xlabel('Training steps')
    

    icl_scores = []
    for checkpoint in checkpoints:
        if checkpoint == 143000:
            scores_dir = f"{outputs_dir}/{model}"
        else:
            scores_dir = f"{outputs_dir}/{model}-{checkpoint}"
        ckpt_scores = []
        for task in tasks:
            if 'icl' in task:
                continue
            
            with open(f"{scores_dir}/{task}_random_0.0.txt", "r") as f:
                scores = list(map(float, f.readlines()))
                ckpt_scores.append(np.mean(scores))
        icl_scores.append(np.mean(ckpt_scores))

    ax.plot(checkpoints, icl_scores, color=colors[5], zorder=1, label='ICL accuracy')

In [ ]:
from matplotlib.ticker import MultipleLocator, LogLocator


plt.figure()
# fig, axs = plt.subplots(3, 3, figsize=(15, 9))
fig, axs = plt.subplots(1, 3, figsize=(13, 2.5))

heads = ['induction', 'fv']
sub_ckpt_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']
ckpt_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']

for ax, model in zip(axs.flat, sub_ckpt_models):
    plot_head_ckpt(model, 'induction', ax)
    ax2 = ax.twinx()
    plot_head_ckpt(model, 'fv', ax2)
    if model == sub_ckpt_models[0]:
        ax.set_ylabel('Induction score', color=colors[2])
    if model == 'pythia-6.9b':
        ax2.set_ylabel('FV score', color=colors[3])
    ax4 = ax.twinx()
    plot_icl(model, ax4)
    # remove ax4 y axis ticks
    ax4.set_yticks([])
    # ax4.tick_params(axis='y', labelcolor=colors[5])
    # ax.xaxis.set_major_locator(LogLocator(base=2, subs=None, numticks=10))
    custom_ticks = [2**i for i in range(0, 18)]
    ax.set_xticks(custom_ticks)
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.05))
    ax.minorticks_on()
    ax.grid(which='both', color='#DDDDDD', linestyle='-', linewidth=0.8)

ax3 = plt.gca()
handles, labels = ax.get_legend_handles_labels()



(zip(labels, handles))
# plt.legend(by_label.values(), by_label.keys())
# ax.legend()
# plt.legend()
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color=colors[3], linestyle='-', label='FV score')
handles.append(custom_handle)
labels.append('FV score')
custom_handle = Line2D([0], [0], color=colors[5], linestyle='-', label='ICL accuracy')
handles.append(custom_handle)
labels.append('ICL accuracy')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.1), ncol=3)
# plt.subplots_adjust(bottom=0.2)
plt.savefig('plots/mean_ckpt_rebut.pdf', bbox_inches='tight')
plt.show()



In [ ]:
plt.figure()
fig, axs = plt.subplots(6, 4, figsize=(15, 18))
# plot_task_ablations('pythia-6.9b', axs)

for ax, (model, excl) in zip(axs.flat, itertools.product(models, [False, True])):
    try:
        if not excl:
            handles, labels = plot_task_ablations(model, ax)
        else:
            plot_task_ablations(model, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
custom_handle = Line2D([0], [0], color='black', linestyle='--', label='Ablation with exclusion')
handles.append(custom_handle)
labels.append('Ablation with exclusion')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=5)
plt.savefig('plots/abl_all.pdf', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()
fig, axs = plt.subplots(3, 3, figsize=(15, 9))
# fig, axs = plt.subplots(1, 3, figsize=(13, 2.5))

heads = ['induction', 'fv']
sub_ckpt_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']
ckpt_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']

for ax, model in zip(axs.flat, ckpt_models):
    plot_head_ckpt(model, 'induction', ax)
    ax2 = ax.twinx()
    plot_head_ckpt(model, 'fv', ax2)
    ax.set_ylabel('Induction score', color=colors[2])
    ax2.set_ylabel('FV score', color=colors[3])
    ax4 = ax.twinx()
    plot_icl(model, ax4)
    ax4.tick_params(axis='y', labelcolor=colors[5])
ax3 = plt.gca()
handles, labels = ax.get_legend_handles_labels()

(zip(labels, handles))
# plt.legend(by_label.values(), by_label.keys())
# ax.legend()
# plt.legend()
custom_handle = Line2D([0], [0], color=colors[3], linestyle='-', label='FV score')
handles.append(custom_handle)
labels.append('FV score')
custom_handle = Line2D([0], [0], color=colors[5], linestyle='-', label='ICL accuracy')
handles.append(custom_handle)
labels.append('ICL accuracy')
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=3)
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
plt.savefig('plots/mean_ckpt_all.pdf', bbox_inches='tight')
plt.show()

In [ ]:
len(colors)

In [ ]:
import random
def plot_icl_task(model, ax, task_idx):
    tasks = test_data#list(map(lambda x: x.replace("_val",""), new_tasks))
    task = tasks[task_idx]
    checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]
    outputs_dir = f"./outputs/scores/"

    cmap = plt.cm.Set2  # Choose the 'viridis' colormap
    colors = cmap(np.linspace(0, 1, 8))

    ax.set_xscale('log')
    ax.set_title(longnames[model])
    ax.set_xlabel('Training steps')
    

    icl_scores = []
    for checkpoint in checkpoints:
        if checkpoint == 143000:
            scores_dir = f"{outputs_dir}/{model}"
        else:
            scores_dir = f"{outputs_dir}/{model}-{checkpoint}"
        ckpt_scores = []
        with open(f"{scores_dir}/{task}_random_0.0.txt", "r") as f:
            scores = list(map(float, f.readlines()))
            ckpt_scores.append(np.mean(scores))
        icl_scores.append(np.mean(ckpt_scores))
    color_id=random.choice([0,1,4,5,6,7])
    ax.plot(checkpoints, icl_scores, color=colors[color_id], zorder=1)
    # annotate line with task name
    x = np.array(checkpoints)
    y = np.array(icl_scores)
    ax.annotate(task, (x[-1], y[-1]), textcoords="offset points", xytext=(0,10), ha='center', fontsize=8, color=colors[color_id])

heads = ['induction', 'fv']
sub_ckpt_models = ['pythia-160m', 'pythia-1b', 'pythia-6.9b']
ckpt_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']

for model in ['pythia-6.9b']:
  
    plt.figure()
    fig, ax = plt.subplots(1, 1, figsize=(20, 10))
    plot_head_ckpt(model, 'induction', ax)
    ax2 = ax.twinx()
    plot_head_ckpt(model, 'fv', ax2)
    ax.set_ylabel('Induction score', color=colors[2])
    ax2.set_ylabel('FV score', color=colors[3])
    ax4 = ax.twinx()
    for i in range(18):
        plot_icl_task(model, ax4, i)
    ax4.tick_params(axis='y', labelcolor=colors[5])
    plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.
    # plt.title(f'{model}')
    # plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.legend(loc='center left')
    plt.savefig(f'plots/abl_{model}.pdf')
    plt.show()
    # break

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]

model = 'pythia-160m'
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

def plot_indv_ckpt(model, score, ax):
    rh = top_heads(get_scores(model, 'induction'))
    np.random.shuffle(rh)
    n = 0.02 * num_heads[model]
    n = np.ceil(n).astype(int)
    n = min(n, 10)
    # n = 10
    rh = rh[:n]
    ih = top_heads(get_scores(model, 'induction'))[:n]
    fv = top_heads(get_scores(model, 'fv'))[:n]
    rih = top_heads(get_scores(model, 'induction'))[:n]
    markers = ['$1$', '$2$', '$3$', '$4$', '$5$', '$6$', '$7$', '$8$', '$9$', '$10$']

    # ax.set_title(longnames[model])
    if score == 'fv':
        ax.set_xlabel('Training steps')
    else:
        ax.set_title(longnames[model])
    if model == sub_models[0]:
        ax.set_ylabel(f"{score.replace('induction', 'Induction').replace('fv', 'FV')} score")
    results = defaultdict(list)
    for ckpt in checkpoints:
        results['ckpt'].append(ckpt)
        if ckpt < 143000:
            scores = get_scores(model, score, ckpt)
        else:
            scores = get_scores(model, score)
        for head in rh:
            # plt.scatter(np.log2(ckpt), scores[head], label='ih', marker='o', color=colors[0])
            results[f'{head}-random'].append(scores[head])
        results['mean'].append(np.mean(list(scores.values())))
        for head in fv:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-fv'].append(scores[head])
        for head in rih:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-rih'].append(scores[head])
    
    # ax.plot(results['ckpt'], results[f'mean'], label=f'mean', marker='', color=colors[2])
    # for i,head in enumerate(rh):
        # ax.plot(results['ckpt'], results[f'{head}-random'], label=f'top {n} random', marker=markers[i], color=colors[0], alpha=1 - (i/(1+n)))
    for i,head in enumerate(rih):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-rih'], label=f'Induction heads', marker='', color=colors[2], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-rih'], marker='', color=colors[2], alpha=1 - (i/(1+n)))
    for i,head in enumerate(fv):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-fv'], label=f'FV heads', marker='',  color=colors[3], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-fv'], marker='', color=colors[3], alpha=1 - (i/(1+n)))
    ax2 = plt.gca()
    ax.set_ylim(bottom=0)
    # if 'induction' in score:
    #     ax.set_ylim(top=0.5)
    # else:
    #     ax.set_ylim(top=0.13)
    # set regular y ticks
    # y_min, y_max = ax.get_ylim()
    # yticks = np.linspace(y_min, y_max, 10)
    # ax.set_yticks(yticks)
    # set y scale to log
    # ax.set_yscale('symlog')
    # ax.set_yticks([])
    # ax.set_yticklabels([])
    ax.set_xscale('log', base=2)
    handles, labels = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    # ax.legend()
    # set minimum value of y axis to 0

    return results

plt.figure()
# fig, axs = plt.subplots(7, 2, figsize=(20, 35))
fig, axs = plt.subplots(2, 3, figsize=(12, 5))
# fig, axs = plt.subplots(1, 1, figsize=(6, 4))

# results = plot_indv_ckpt('pythia-6.9b', 'induction', axs)

scores = ['induction', 'fv']
ckpt_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b', 'pythia-6.9b']
for ax, (score, model) in zip(axs.flat, itertools.product(scores, sub_ckpt_models)):
    try:
        results = plot_indv_ckpt(model, score, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
# for ax, (model, score) in zip(axs.flat, itertools.product(ckpt_models, scores)):
#     try:
#         results = plot_indv_ckpt(model, score, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.legend(fontsize='small')
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.

plt.savefig('plots/indv_ckpt.pdf')

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]

model = 'pythia-6.9b'
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

n = 1
fv = top_heads(get_scores(model, 'fv'))[:n]
markers = ['$1$', '$2$', '$3$', '$4$', '$5$', '$6$', '$7$', '$8$', '$9$', '$10$']

fig, ax = plt.subplots()
score = 'induction'
ax.set_xlabel('Training steps')
ax.set_ylabel('Induction score', color=colors[2])
results = defaultdict(list)
for ckpt in checkpoints:
    results['ckpt'].append(ckpt)
    if ckpt < 143000:
        scores = get_scores(model, score, ckpt)
    else:
        scores = get_scores(model, score)
    for head in fv:
        # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
        results[f'{head}-fv'].append(scores[head])


for i,head in enumerate(fv):
    if i == 0:
        ax.plot(results['ckpt'], results[f'{head}-fv'], label=f'FV', marker='', linestyle=':', color=colors[2], alpha=1 - (i/(1+n)))
    else:
        ax.plot(results['ckpt'], results[f'{head}-fv'], marker='', linestyle=':', color=colors[2], alpha=1 - (i/(1+n)))
ax3 = ax.twinx()
ax3.set_ylabel('FV score', color=colors[3])
score = 'fv'
results = defaultdict(list)
for ckpt in checkpoints:
    results['ckpt'].append(ckpt)
    if ckpt < 143000:
        scores = get_scores(model, score, ckpt)
    else:
        scores = get_scores(model, score)
    for head in fv:
        # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
        results[f'{head}-fv'].append(scores[head])

for i,head in enumerate(fv):
    if i == 0:
        ax.plot(results['ckpt'], results[f'{head}-fv'], label=f'FV', marker='', color=colors[3], alpha=1 - (i/(1+n)))
    else:
        ax.plot(results['ckpt'], results[f'{head}-fv'], marker='', color=colors[3], alpha=1 - (i/(1+n)))


ax2 = plt.gca()
ax.set_ylim(bottom=0)
# if 'induction' in score:
#     ax.set_ylim(top=0.5)
# else:
#     ax.set_ylim(top=0.13)
# set regular y ticks
# y_min, y_max = ax.get_ylim()
# yticks = np.linspace(y_min, y_max, 10)
# ax.set_yticks(yticks)
# set y scale to log
# ax.set_yscale('symlog')
# ax.set_yticks([])
# ax.set_yticklabels([])
ax.set_xscale('log', base=2)
handles, labels = ax2.get_legend_handles_labels()
by_label = dict(zip(labels, handles))


plt.figure()
# fig, axs = plt.subplots(7, 2, figsize=(20, 35))

plt.legend()
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.

plt.savefig('plots/indv_ckpt_main.pdf')

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]

model = 'pythia-160m'
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

def plot_indv_ckpt(model, score, ax):
    rh = top_heads(get_scores(model, 'induction'))
    np.random.shuffle(rh)
    n = 0.02 * num_heads[model]
    n = np.ceil(n).astype(int)
    n = min(n, 10)
    n = 10
    rh = rh[:n]
    ih = top_heads(get_scores(model, 'induction'))[:n]
    fv = top_heads(get_scores(model, 'fv'))[:n]
    rih = top_heads(get_scores(model, 'induction'))[:n]
    markers = ['$1$', '$2$', '$3$', '$4$', '$5$', '$6$', '$7$', '$8$', '$9$', '$10$']

    # ax.set_title(longnames[model])
    ax.set_xlabel('Steps')
    ax.set_title(longnames[model])
    ax.set_ylabel(f"{score.replace('induction', 'Induction').replace('fv', 'FV')} score")
    results = defaultdict(list)
    for ckpt in checkpoints:
        results['ckpt'].append(ckpt)
        if ckpt < 143000:
            scores = get_scores(model, score, ckpt)
        else:
            scores = get_scores(model, score)
        for head in rh:
            # plt.scatter(np.log2(ckpt), scores[head], label='ih', marker='o', color=colors[0])
            results[f'{head}-random'].append(scores[head])
        results['mean'].append(np.mean(list(scores.values())))
        for head in fv:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-fv'].append(scores[head])
        for head in rih:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-rih'].append(scores[head])
    
    # ax.plot(results['ckpt'], results[f'mean'], label=f'mean', marker='', color=colors[2])
    # for i,head in enumerate(rh):
        # ax.plot(results['ckpt'], results[f'{head}-random'], label=f'top {n} random', marker=markers[i], color=colors[0], alpha=1 - (i/(1+n)))
    for i,head in enumerate(rih):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-rih'], label=f'Induction', marker='', color=colors[2], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-rih'], marker='', color=colors[2], alpha=1 - (i/(1+n)))
    for i,head in enumerate(fv):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-fv'], label=f'FV', marker='', color=colors[3], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-fv'], marker='', color=colors[3], alpha=1 - (i/(1+n)))
    ax2 = plt.gca()
    ax.set_ylim(bottom=0)
    # if 'induction' in score:
    #     ax.set_ylim(top=0.5)
    # else:
    #     ax.set_ylim(top=0.13)
    # set regular y ticks
    # y_min, y_max = ax.get_ylim()
    # yticks = np.linspace(y_min, y_max, 10)
    # ax.set_yticks(yticks)
    # set y scale to log
    # ax.set_yscale('symlog')
    # ax.set_yticks([])
    # ax.set_yticklabels([])
    ax.set_xscale('log', base=2)
    handles, labels = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    # ax.legend()
    # set minimum value of y axis to 0

    return results

plt.figure()
fig, axs = plt.subplots(7, 2, figsize=(16, 28))
# fig, axs = plt.subplots(4, 3, figsize=(12, 10))
# fig, axs = plt.subplots(1, 1, figsize=(6, 4))

# results = plot_indv_ckpt('pythia-6.9b', 'induction', axs)

scores = ['induction', 'fv']
ckpt_models = ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b', 'pythia-2.8b']
for ax, (model, score) in zip(axs.flat, itertools.product(pythia_models, scores)):
    try:
        results = plot_indv_ckpt(model, score, ax)
    except Exception as e:
        print(e)
        print(model)
        ax.set_title(f'{model} missing')
# for ax, (model, score) in zip(axs.flat, itertools.product(ckpt_models, scores)):
#     try:
#         results = plot_indv_ckpt(model, score, ax)
#     except Exception as e:
#         print(e)
#         print(model)
#         ax.set_title(f'{model} missing')
plt.legend()
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.

plt.savefig('plots/indv_ckpt_all.pdf')

In [ ]:
checkpoints = [1, 64, 256, 1000, 4000, 16000, 64000, 143000]

model = 'pythia-160m'
cmap = plt.cm.Set2 
colors = cmap(np.linspace(0, 1, 8))

def plot_indv_ckpt(model, score, ax):
    rh = top_heads(get_scores(model, 'induction'))
    np.random.shuffle(rh)
    n = 0.02 * num_heads[model]
    n = np.ceil(n).astype(int)
    n = 10
    rh = rh[:n]
    ih = top_heads(get_scores(model, 'induction'))[:n]
    fv = top_heads(get_scores(model, 'fv'))[:n]
    rih = top_heads(get_scores(model, 'induction'))[:n]
    markers = ['$1$', '$2$', '$3$', '$4$', '$5$', '$6$', '$7$', '$8$', '$9$', '$10$']

    # ax.set_title(longnames[model])
    ax.set_xlabel('Steps')

    ax.set_ylabel(f"{score.replace('induction', 'Induction').replace('fv', 'FV')} score")
    results = defaultdict(list)
    for ckpt in checkpoints:
        results['ckpt'].append(ckpt)
        if ckpt < 143000:
            scores = get_scores(model, score, ckpt)
        else:
            scores = get_scores(model, score)
        for head in rh:
            # plt.scatter(np.log2(ckpt), scores[head], label='ih', marker='o', color=colors[0])
            results[f'{head}-random'].append(scores[head])
        results['mean'].append(np.mean(list(scores.values())))
        for head in fv:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-fv'].append(scores[head])
        for head in rih:
            # plt.scatter(np.log2(ckpt), scores[head], label='fv', marker='x', color=colors[1])
            results[f'{head}-rih'].append(scores[head])
    
    # ax.plot(results['ckpt'], results[f'mean'], label=f'mean', marker='', color=colors[2])
    # for i,head in enumerate(rh):
        # ax.plot(results['ckpt'], results[f'{head}-random'], label=f'top {n} random', marker=markers[i], color=colors[0], alpha=1 - (i/(1+n)))
    for i,head in enumerate(rih):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-rih'], label=f'Induction', marker='', color=colors[2], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-rih'], marker='', color=colors[2], alpha=1 - (i/(1+n)))
    for i,head in enumerate(fv):
        if i == 0:
            ax.plot(results['ckpt'], results[f'{head}-fv'], label=f'FV', marker='', linestyle=':', color=colors[3], alpha=1 - (i/(1+n)))
        else:
            ax.plot(results['ckpt'], results[f'{head}-fv'], marker='', linestyle=':', color=colors[3], alpha=1 - (i/(1+n)))
    ax2 = plt.gca()
    ax.set_ylim(bottom=0)
    # if 'induction' in score:
    #     ax.set_ylim(top=0.5)
    # else:
    #     ax.set_ylim(top=0.13)
    # set regular y ticks
    # y_min, y_max = ax.get_ylim()
    # yticks = np.linspace(y_min, y_max, 10)
    # ax.set_yticks(yticks)
    # set y scale to log
    # ax.set_yscale('symlog')
    # ax.set_yticks([])
    # ax.set_yticklabels([])
    ax.set_xscale('log', base=2)
    handles, labels = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    # ax.legend()
    # set minimum value of y axis to 0

    return results

plt.figure()
# fig, axs = plt.subplots(7, 2, figsize=(20, 35))
# fig, axs = plt.subplots(2, 3, figsize=(12, 5))
fig, axs = plt.subplots(1, 1, figsize=(6, 4))

results = plot_indv_ckpt('pythia-6.9b', 'induction', axs)

plt.legend()
plt.tight_layout()  # Adjusts subplot params so that subplots fit into the figure area.

plt.savefig('plots/indv_ckpt_1.pdf')

In [ ]:
fv #2,3


In [ ]:
rih